In [ ]:
!git clone https://github.com/im-xiaoming/MMDF_.git
!pip install "transformers<5.0"


In [ ]:
%%writefile train_v2_script.py
"""
TRAIN SCRIPT - PHUONG PHAP 2 (RNN_HYBRID) - MULTI-GPU.
Phien ban x2 GPU dung `accelerate launch --multi_gpu`.
Tuong duong train_x2_gpu.ipynb nhung su dung mo hinh moi RNN_HYBRID va config v2.
"""
from ruamel.yaml import YAML
import numpy as np
import os
from collections import OrderedDict

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.backends.cudnn as cudnn

from transformers import BertTokenizerFast
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, roc_curve
from scipy.optimize import brentq
from scipy.interpolate import interp1d

import MMDF_.utils as utils
from MMDF_.dataset import create_dataset, create_loader
from MMDF_.scheduler import create_scheduler
from MMDF_.optim import create_optimizer
from MMDF_.utils import text_input_adjust, AttrDict
from MMDF_.models import box_ops
from MMDF_.tools.multilabel_metrics import AveragePrecisionMeter, get_multi_label

# === Mo hinh moi (phuong phap 2) ===
from MMDF_.models_v2 import RNN_HYBRID

from accelerate import Accelerator


def train(args, model, data_loader, optimizer, tokenizer, epoch, EPOCH,
          warmup_steps, device, scheduler, config, accelerator):
    model.train()

    step_size = 100
    warmup_iterations = warmup_steps * step_size
    avg_loss = []

    pbar = tqdm(data_loader, disable=not accelerator.is_main_process, ncols=200)
    for i, (image, label, text, fake_image_box, fake_word_pos, W, H) in enumerate(pbar):
        pbar.set_description(f"[v2] Epoch {epoch}/{EPOCH}")

        if config['schedular']['sched'] == 'cosine_in_step':
            scheduler.adjust_learning_rate(optimizer, i / len(data_loader) + epoch, args, config)

        optimizer.zero_grad()

        image = image.to(device, non_blocking=True)
        fake_image_box = fake_image_box.to(device)

        text_input = tokenizer(text, max_length=128, truncation=True, add_special_tokens=True,
                               return_attention_mask=True, return_token_type_ids=False)
        text_input, fake_token_pos = text_input_adjust(text_input, fake_word_pos, device)

        if epoch > 0:
            alpha = config['alpha']
        else:
            alpha = config['alpha'] * min(1, i / len(data_loader))

        loss_MAC, loss_BIC, loss_bbox, loss_giou, loss_TMG, loss_MLC = model(
            image, label, text_input, fake_image_box, fake_token_pos, alpha=alpha
        )

        loss = (config['loss_MAC_wgt'] * loss_MAC
                + config['loss_BIC_wgt'] * loss_BIC
                + config['loss_bbox_wgt'] * loss_bbox
                + config['loss_giou_wgt'] * loss_giou
                + config['loss_TMG_wgt'] * loss_TMG
                + config['loss_MLC_wgt'] * loss_MLC)

        avg_loss.append(loss.item())

        accelerator.backward(loss)
        optimizer.step()

        pbar.set_postfix(OrderedDict([
            ('MAC', f'{loss_MAC.item():.3f}'),
            ('BIC', f'{loss_BIC.item():.3f}'),
            ('bbox', f'{loss_bbox.item():.3f}'),
            ('giou', f'{loss_giou.item():.3f}'),
            ('TMG', f'{loss_TMG.item():.3f}'),
            ('MLC', f'{loss_MLC.item():.3f}'),
            ('loss', f'{loss.item():.3f}'),
            ('AVG', f'{np.mean(avg_loss):.3f}'),
            ('lr', f"{optimizer.param_groups[0]['lr']}"),
        ]))

        if epoch == 0 and i % step_size == 0 and i <= warmup_iterations \
                and config['schedular']['sched'] != 'cosine_in_step':
            scheduler.step(i // step_size)


@torch.no_grad()
def evaluate_v2(model, data_loader, tokenizer, device):
    """Eval day du cho RNN_HYBRID (khong dung evaluate() goc - kia danh cho HAMMER)."""
    model.eval()

    y_true, y_pred, IOU_pred, IOU_50, IOU_75, IOU_95 = [], [], [], [], [], []
    cls_nums_all, cls_acc_all = 0, 0
    TP_all = TN_all = FP_all = FN_all = 0
    TP_m = np.zeros(4, dtype=int)
    FP_m = np.zeros(4, dtype=int)
    FN_m = np.zeros(4, dtype=int)
    F1_m = np.zeros(4)

    meter = AveragePrecisionMeter(difficult_examples=False)
    meter.reset()

    for image, label, text, fake_image_box, fake_word_pos, W, H in tqdm(data_loader, ncols=120):
        image = image.to(device, non_blocking=True)
        text_input = tokenizer(text, max_length=128, truncation=True, add_special_tokens=True,
                               return_attention_mask=True, return_token_type_ids=False)
        text_input, fake_token_pos = text_input_adjust(text_input, fake_word_pos, device)

        logits_rf, logits_mc, output_coord, logits_tok = model(
            image, label, text_input, fake_image_box, fake_token_pos, is_train=False
        )

        cls_label = torch.ones(len(label), dtype=torch.long, device=device)
        real_pos = np.where(np.array(label) == 'orig')[0].tolist()
        cls_label[real_pos] = 0
        y_pred.extend(F.softmax(logits_rf, dim=1)[:, 1].cpu().tolist())
        y_true.extend(cls_label.cpu().tolist())
        cls_acc_all += (logits_rf.argmax(1) == cls_label).sum().item()
        cls_nums_all += cls_label.size(0)

        target, _ = get_multi_label(label, image)
        meter.add(logits_mc, target)
        for c in range(logits_mc.shape[1]):
            p = logits_mc[:, c].clone()
            p = (p >= 0).long()
            TP_m[c] += ((target[:, c] == 1) * (p == 1)).sum().item()
            FP_m[c] += ((target[:, c] == 0) * (p == 1)).sum().item()
            FN_m[c] += ((target[:, c] == 1) * (p == 0)).sum().item()

        b1 = box_ops.box_cxcywh_to_xyxy(output_coord)
        b2 = box_ops.box_cxcywh_to_xyxy(fake_image_box.to(device))
        IOU, _ = box_ops.box_iou(b1, b2, test=True)
        IOU_pred.extend(IOU.cpu().tolist())
        IOU_50.extend((IOU > 0.5).long().cpu().tolist())
        IOU_75.extend((IOU > 0.75).long().cpu().tolist())
        IOU_95.extend((IOU > 0.95).long().cpu().tolist())

        token_label = text_input.attention_mask[:, 1:].clone()
        token_label[token_label == 0] = -100
        token_label[token_label == 1] = 0
        for b_idx, positions in enumerate(fake_token_pos):
            for pos in positions:
                if pos < token_label.size(1):
                    token_label[b_idx, pos] = 1
        logits_tok_a = logits_tok[:, 1:, :]
        L = min(logits_tok_a.size(1), token_label.size(1))
        pred_tok = logits_tok_a[:, :L, :].argmax(-1).reshape(-1)
        tlab = token_label[:, :L].reshape(-1)
        m = tlab != -100
        pred_tok, tlab = pred_tok[m], tlab[m]
        TP_all += ((tlab == 1) * (pred_tok == 1)).sum().item()
        TN_all += ((tlab == 0) * (pred_tok == 0)).sum().item()
        FP_all += ((tlab == 0) * (pred_tok == 1)).sum().item()
        FN_all += ((tlab == 1) * (pred_tok == 0)).sum().item()

    y_true, y_pred = np.array(y_true), np.array(y_pred)
    AUC = roc_auc_score(y_true, y_pred) if len(set(y_true)) > 1 else float('nan')
    ACC = cls_acc_all / max(cls_nums_all, 1)
    try:
        fpr, tpr, _ = roc_curve(y_true, y_pred, pos_label=1)
        EER = brentq(lambda x: 1. - x - interp1d(fpr, tpr)(x), 0., 1.)
    except Exception:
        EER = float('nan')

    IOU_score = sum(IOU_pred) / max(len(IOU_pred), 1)
    IOU50 = sum(IOU_50) / max(len(IOU_50), 1)
    IOU75 = sum(IOU_75) / max(len(IOU_75), 1)
    IOU95 = sum(IOU_95) / max(len(IOU_95), 1)

    ACC_tok = (TP_all + TN_all) / max(TP_all + TN_all + FP_all + FN_all, 1)
    P_tok = TP_all / max(TP_all + FP_all, 1)
    R_tok = TP_all / max(TP_all + FN_all, 1)
    F1_tok = 2 * P_tok * R_tok / max(P_tok + R_tok, 1e-8)

    MAP = meter.value().mean().item()
    OP, OR, OF1, CP, CR, CF1 = meter.overall()
    for c in range(4):
        P = TP_m[c] / max(TP_m[c] + FP_m[c], 1)
        R = TP_m[c] / max(TP_m[c] + FN_m[c], 1)
        F1_m[c] = 2 * P * R / max(P + R, 1e-8)

    return {
        'AUC_cls': f'{AUC*100:.4f}', 'ACC_cls': f'{ACC*100:.4f}',
        'EER_cls': f'{EER*100:.4f}', 'MAP': f'{MAP*100:.4f}',
        'OP': f'{OP*100:.4f}', 'OR': f'{OR*100:.4f}', 'OF1': f'{OF1*100:.4f}',
        'CP': f'{CP*100:.4f}', 'CR': f'{CR*100:.4f}', 'CF1': f'{CF1*100:.4f}',
        'F1_FS': f'{F1_m[0]*100:.4f}', 'F1_FA': f'{F1_m[1]*100:.4f}',
        'F1_TS': f'{F1_m[2]*100:.4f}', 'F1_TA': f'{F1_m[3]*100:.4f}',
        'IOU_score': f'{IOU_score*100:.4f}',
        'IOU_ACC_50': f'{IOU50*100:.4f}', 'IOU_ACC_75': f'{IOU75*100:.4f}',
        'IOU_ACC_95': f'{IOU95*100:.4f}',
        'ACC_tok': f'{ACC_tok*100:.4f}',
        'Precision_tok': f'{P_tok*100:.4f}', 'Recall_tok': f'{R_tok*100:.4f}',
        'F1_tok': f'{F1_tok*100:.4f}',
    }


def main_run():
    args = {
        'config': 'MMDF_/configs/train_v2.yaml',     # config v2
        'checkpoint': '',                            # de '' neu train tu dau
        'resume': False,
        'output_dir': 'results_v2',
        'text_encoder': 'bert-base-uncased',         # chi dung cho tokenizer
        'device': 'cuda',
        'world_size': 1,
        'launcher': 'pytorch',
        'model_save_epoch': 1,
        'token_momentum': False,
    }
    args = AttrDict(args)
    yaml = YAML(typ='safe')

    with open(args.config, 'r') as f:
        config = yaml.load(f)

    accelerator = Accelerator()
    device = accelerator.device
    cudnn.benchmark = True

    start_epoch = 0
    max_epoch = config['schedular']['epochs']
    warmup_steps = config['schedular']['warmup_epochs']

    train_dataset, val_dataset = create_dataset(config)
    train_loader, val_loader = create_loader(
        [train_dataset, val_dataset],
        batch_size=[config['batch_size_train']] + [config['batch_size_val']],
        num_workers=[4, 4],
        is_trains=[True, False],
    )

    tokenizer = BertTokenizerFast.from_pretrained(args.text_encoder)

    model = RNN_HYBRID(args=args, config=config, text_encoder=args.text_encoder,
                       tokenizer=tokenizer, init_deit=True)
    model = model.to(device)

    arg_opt = utils.AttrDict(config['optimizer'])
    optimizer = create_optimizer(arg_opt, model)
    arg_sche = utils.AttrDict(config['schedular'])
    lr_scheduler, _ = create_scheduler(arg_sche, optimizer)
    if config['schedular']['sched'] == 'cosine_in_step':
        args.lr = config['optimizer']['lr']

    # load checkpoint (tuy chon - khong dung load_checkpoint goc vi no co logic
    # rieng cho visual_encoder.pos_embed cua HAMMER)
    if args.checkpoint:
        ckpt = torch.load(args.checkpoint, map_location='cpu', weights_only=False)
        state = ckpt.get('model', ckpt)
        msg = model.load_state_dict(state, strict=False)
        if accelerator.is_main_process:
            print('[v2] load ckpt:', msg)
        if args.resume and 'optimizer' in ckpt:
            optimizer.load_state_dict(ckpt['optimizer'])
            start_epoch = ckpt.get('epoch', 0)
            if accelerator.is_main_process:
                print(f'[v2] Resume from epoch {start_epoch}')

    model, optimizer, train_loader, lr_scheduler = accelerator.prepare(
        model, optimizer, train_loader, lr_scheduler
    )

    os.makedirs('results_v2', exist_ok=True)
    os.makedirs('checkpoints_v2', exist_ok=True)

    for epoch in range(start_epoch, 40):
        if accelerator.is_main_process:
            print(f"\n--- [v2] Epoch {epoch+1} ---")
            print("train\n")

        train(args, model, train_loader, optimizer, tokenizer, epoch, max_epoch,
              warmup_steps, device, lr_scheduler, config, accelerator)

        accelerator.wait_for_everyone()

        if accelerator.is_main_process:
            print("val\n")
            raw_model = accelerator.unwrap_model(model)
            val_stats = evaluate_v2(raw_model, val_loader, tokenizer, device)
            print(val_stats)

            with open(os.path.join('results_v2', f"result_{epoch+1}.txt"), 'w') as f:
                f.write(str(val_stats))

            torch.save(
                {'model': raw_model.state_dict(),
                 'optimizer': optimizer.state_dict(),
                 'epoch': epoch + 1,
                 'config': config,
                 'val_stats': val_stats},
                os.path.join('checkpoints_v2', f'rnn_hybrid_{epoch+1:02d}.pth'),
            )

        accelerator.wait_for_everyone()


if __name__ == '__main__':
    main_run()


In [ ]:
!accelerate launch --multi_gpu --num_processes 2 --mixed_precision fp16 train_v2_script.py
